# Used-Car Price Prediction: Exploratory Data Analysis

This notebook is intentionally focused on exploration and modelling decisions. The reusable preprocessing and training logic lives in `src/` so the notebook is not a hidden production dependency.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
DATA_PATH = Path('../data/autoscout_car_sales.csv')
df = pd.read_csv(DATA_PATH)
df.shape, df.head()

## Dataset quality and target
Check types, duplicates, missingness and the target range before drawing conclusions.

In [ ]:
display(df.dtypes.rename('dtype').to_frame())
display(df.isna().sum().sort_values(ascending=False).rename('missing').to_frame().head(10))
print(f'Duplicate rows: {df.duplicated().sum():,}')
display(df['price'].describe().to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['price'], kde=True, ax=axes[0])
axes[0].set_title('Price in euros')
sns.histplot(np.log1p(df['price']), kde=True, ax=axes[1])
axes[1].set_title('log1p(price)')
plt.tight_layout();

The raw target has a long right tail. Training on `log1p(price)` reduces the influence of high-priced listings and makes multiplicative pricing relationships easier for a linear model to represent. Evaluation must still be reported after converting predictions back to euros.

## Numerical relationships and categorical segments

In [ ]:
numeric = df.select_dtypes(include='number')
plt.figure(figsize=(10, 7))
sns.heatmap(numeric.corr(), cmap='vlag', center=0)
plt.title('Numerical correlation matrix')
plt.tight_layout();

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.scatterplot(data=df.sample(4000, random_state=42), x='age', y='price', hue='make_model', alpha=.35, legend=False, ax=axes[0])
sns.scatterplot(data=df.sample(4000, random_state=42), x='km', y='price', hue='make_model', alpha=.35, legend=False, ax=axes[1])
axes[0].set_title('Price vs vehicle age')
axes[1].set_title('Price vs mileage')
plt.tight_layout();

In [ ]:
segment_summary = (df.groupby('make_model')['price']
                     .agg(listings='size', median_price='median', mean_price='mean')
                     .sort_values('median_price', ascending=False))
display(segment_summary)

## Equipment-list feature engineering

In [ ]:
equipment_columns = ['Comfort_Convenience', 'Entertainment_Media', 'Extras', 'Safety_Security']
equipment_counts = pd.DataFrame({
    f'num_{column.lower()}': df[column].fillna('').map(lambda value: len({x.strip() for x in value.split(',') if x.strip()}))
    for column in equipment_columns
})
display(equipment_counts.describe().T)
display(equipment_counts.assign(price=df['price']).corr()['price'].sort_values(ascending=False).to_frame())

## Modelling decisions

- Remove exact duplicates, use a grouped 80/20 holdout, and keep identical predictor rows in one partition.
- Choose hyperparameters only by grouped cross-validation on the training split.
- Put imputation, IQR clipping, scaling and encoding inside the pipeline to prevent leakage.
- Represent equipment lists with counts for stable inference on unseen accessories.
- Compare OLS, Ridge, Lasso and Elastic Net on MAE in euros.
- Treat R-squared as secondary and inspect residuals for missing nonlinear structure.

Run `python -m src.train` from the project root for the authoritative model comparison.